In [1]:
import rectanglepy as rectangle
import pandas as pd
import anndata as ad
import pickle

In [2]:
messmer = pd.read_table('/nfs/home/students/t.tabanli/artifical_meat_deconv/messmer_counts/GSE211428_counts.tsv.gz', index_col=0, sep='\t')
col_meta = pd.read_table('/nfs/data2/huji_artificial_meat/deconvolution_data/external_sc_references/messmer2023_GSE211428/GSE211428_combined_meta.tsv.gz', sep = '\t')

col_meta['seurat_barcode'] = col_meta['Unique_barcode'].str.replace(
    r'^(.*)\.([0-9]+)_([0-9]+)$', r'Inj\3_\1-\2', regex=True)
col_meta = col_meta.set_index('seurat_barcode').drop(columns=['V1'], errors='ignore')
col_meta['predicted.pheno'] = col_meta['predicted.pheno'].str.replace('/', '-')

print(f"mat columns: {messmer.shape[1]}")
print(f"col_meta barcodes: {len(col_meta)}")
common = messmer.columns.intersection(col_meta.index)
print(f"Intersection: {len(common)}")
print(f"In mat not in col_meta: {len(messmer.columns.difference(col_meta.index))}")
print(f"In col_meta not in mat: {len(col_meta.index.difference(messmer.columns))}")

messmer = messmer[common]
col_meta = col_meta.loc[common]

mat columns: 36140
col_meta barcodes: 36129
Intersection: 36118
In mat not in col_meta: 22
In col_meta not in mat: 11


In [4]:
# remove tenocytes from the reference
tenocytes = col_meta[col_meta['predicted.pheno'] == 'Tenocytes']
col_meta = col_meta[~col_meta.index.isin(tenocytes.index)]
messmer = messmer.drop(columns=tenocytes.index)

In [ ]:
# bulk matrices
bulk_adipo = pd.read_csv('/nfs/data2/huji_artificial_meat/adipogenesis/adipogenesis_tpm.tsv', index_col=0, sep='\t')
bulk_myo   = pd.read_csv('/nfs/data2/huji_artificial_meat/myogenesis/nfcore_out/star_salmon/salmon.merged.gene_tpm.tsv', index_col=0, sep='\t')
bulk_myo = bulk_myo.drop(columns=['gene_name'])

# metadata
meta_adipo = pd.read_csv('/nfs/data2/huji_artificial_meat/adipogenesis/sample_sheet_cleaned.csv', index_col=0)
meta_myo   = pd.read_csv('/nfs/data2/huji_artificial_meat/myogenesis/sample_sheet_cleaned.csv', index_col=0)

meta_adipo.columns = meta_adipo.columns.str.lower().str.replace(r'[^a-z0-9]', '_', regex=True).str.strip('_')
meta_myo.columns   = meta_myo.columns.str.lower().str.replace(r'[^a-z0-9]', '_', regex=True).str.strip('_')

meta_adipo = meta_adipo[['library_name', 'sample_id', 'title', 'group', 'bio_replicate', 'individual', 'sex', 'treatment', 'day']]
meta_adipo['day'] = pd.Categorical(meta_adipo['day'], categories=['D0', 'D2', 'D3', 'D6', 'D14'], ordered=True)
meta_adipo['short_id'] = meta_adipo['library_name'].str.replace("Sample ", "S", regex=True)

meta_myo = meta_myo[['library_name', 'sample_id', 'group', 'day', 'bio_replicate', 'sex', 'treatment']]
meta_myo['day']       = pd.Categorical(meta_myo['day'], categories=['d0', 'd3', 'd7', 'd21'], ordered=True)
meta_myo['group']     = meta_myo['group'].str.replace(r'(?!Sat)D\d*', '', regex=True).str.split('_').str[-1]

# adipo: convert the colnames to samples id by matching to "title" in col_meta
#title_to_id = meta_adipo.set_index('sample_id')['short_id']
#bulk_adipo.columns = bulk_adipo.columns.map(title_to_id)

In [29]:
#help(rectangle.pp.build_rectangle_signatures)
signature_result = rectangle.pp.build_rectangle_signatures(sc_adata)

Help on function build_rectangle_signatures in module rectanglepy.pp.create_signature:

build_rectangle_signatures(adata: anndata._core.anndata.AnnData, cell_type_col: str = 'cell_type', bulks: pandas.core.frame.DataFrame = None, *, optimize_cutoffs=True, layer: str = None, raw: bool = False, p=0.015, lfc=1.5, n_cpus: int = None, gene_expression_threshold=0.5) -> rectanglepy.pp.rectangle_signature.RectangleSignatureResult
    Builds rectangle signatures based on single-cell  count data and annotations.

    Parameters
    ----------
    adata
        The single-cell count data as a DataFrame. DataFrame must have the genes as index and cell identifier as columns. Each entry should be in raw counts.
    bulks
        The bulk data as a DataFrame. DataFrame must have the bulk identifier as index and the genes as columns. Each entry should be in transcripts per million (TPM).
    cell_type_col
        The annotations corresponding to the single-cell count data. Series data should have the 

KeyboardInterrupt: 

In [ ]:
sc_adata = ad.AnnData(messmer.T, obs=col_meta['predicted.pheno'].to_frame(name="cell_type"))
signature_result = rectangle.pp.build_rectangle_signatures()
est_adipo, signature_result = rectangle.rectangle(sc_adata, bulk_adipo.T)

2026-07-06 12:07:53.088 | INFO     | rectanglepy.pp.create_signature:build_rectangle_signatures:329 - Using 15056 common genes between bulks and single-cell data
2026-07-06 12:08:38.615 | INFO     | rectanglepy.pp.create_signature:_de_analysis:202 - Starting DE analysis
2026-07-06 12:08:52.984 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for B-T-NK cells
2026-07-06 12:09:39.579 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for FAPs
2026-07-06 12:09:54.063 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for Glial cells
2026-07-06 12:09:58.790 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for LECs
2026-07-06 12:10:05.988 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for Monocytes - M?
2026-07-06 12:10:10.037 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for Myocytes
2026

Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optima

2026-07-06 12:10:40.941 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:418 - RMSE:0.08313756985607616, Pearson R:0.5594754358281938 for p=0.05, lfc=1.6
2026-07-06 12:10:41.121 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:227 - Optimal condition number: 26


Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point


2026-07-06 12:10:41.463 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:418 - RMSE:0.08313756985607616, Pearson R:0.5594754358281938 for p=0.05, lfc=1.7


...

2026-07-06 12:10:43.584 | INFO     | rectanglepy.pp.create_signature:_optimize_parameters:418 - RMSE:0.08045174154237714, Pearson R:0.5628895579314575 for p=0.05, lfc=2.2
2026-07-06 12:10:43.610 | INFO     | rectanglepy.pp.create_signature:_de_analysis:210 - Optimization done
 Best cutoffs  p: 0.05 and lfc: 2.2


Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point


2026-07-06 12:10:43.754 | INFO     | rectanglepy.pp.create_signature:_get_marker_genes:227 - Optimal condition number: 23
2026-07-06 12:10:43.758 | INFO     | rectanglepy.pp.create_signature:build_rectangle_signatures:350 - Starting rectangle cluster analysis
2026-07-06 12:10:43.971 | INFO     | rectanglepy.pp.create_signature:_create_clustered_data:268 - Starting clustered analysis
2026-07-06 12:11:06.891 | INFO     | rectanglepy.pp.create_signature:_de_analysis:202 - Starting DE analysis
2026-07-06 12:11:26.941 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for 1
2026-07-06 12:11:40.108 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for 2
2026-07-06 12:11:53.873 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for B-T-NK cells
2026-07-06 12:11:55.886 | INFO     | rectanglepy.pp.create_signature:_run_deseq2:150 - Running DE analysis for Glial cells
2026-07-06 12:12:00.719 | INFO  

Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optima

2026-07-06 12:12:31.761 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 4
2026-07-06 12:12:31.925 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 20
2026-07-06 12:12:32.044 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 32
2026-07-06 12:12:32.070 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 28
2026-07-06 12:12:32.087 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 17
2026-07-06 12:12:32.130 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 25
2026-07-06 12:12:32.247 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 1
2026-07-06 12:12:32.274 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 8
2026-07-06 12:12:32.303 | I

In [20]:
# save results as pickle
with open('rectangle_results/rectangle_signature.pkl', 'wb') as f:
    pickle.dump(signature_result, f)

In [25]:
# for myo
est_myo, _ = rectangle.tl.deconvolution(signature_result, bulk_myo.T)

2026-07-06 12:40:47.199 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 28
2026-07-06 12:40:47.201 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 2
2026-07-06 12:40:47.201 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 27
2026-07-06 12:40:47.201 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 58
2026-07-06 12:40:47.203 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 38
2026-07-06 12:40:47.205 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 47
2026-07-06 12:40:47.206 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 9
2026-07-06 12:40:47.207 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:284 - Deconvolute fractions for bulk: 41
2026-07-06 12:40:47.207 | INFO    

Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optimal point
Polishing not needed - no active set detected at optima

2026-07-06 12:41:02.715 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 54
2026-07-06 12:41:03.001 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 48
2026-07-06 12:41:03.043 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 49
2026-07-06 12:41:03.110 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 52
2026-07-06 12:41:03.295 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 43
2026-07-06 12:41:03.584 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 42
2026-07-06 12:41:03.630 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 6
2026-07-06 12:41:04.033 | INFO     | rectanglepy.tl.deconvolution:_process_bulk:287 - Finished deconvolution for bulk: 7
2026-07-06 12:41:04.328 | 

In [ ]:
# also as CSV
est_adipo.reset_index().to_csv('rectangle_results/rectangle_estimations_adipo.csv', index=False)
est_myo.reset_index().to_csv('rectangle_results/rectangle_estimations_myo.csv', index=False)

In [2]:
# read results
#import pickle
with open('rectangle_results/rectangle_signature.pkl', 'rb') as f:
    signature_result = pickle.load(f)

with open('rectangle_results/rectangle_estimations.pkl', 'rb') as f:
    estimations = pickle.load(f)